# Tool

In [6]:
def choose_from_list(options, prompt, invalid_prompt):
    """
    Lets the user choose either:
    - a number like 1, 2, 3...
    - or the exact option name
    """

    options = list(options)

    while True:
        user_input = input(prompt).strip()

        # forced exit
        if user_input == "":
            return None

        # numeric choice
        if user_input.isdigit():
            choice_index = int(user_input) - 1

            if 0 <= choice_index < len(options):
                return options[choice_index]

        # text choice
        if user_input in options:
            return user_input

        print()
        print(invalid_prompt)

def main():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    pd.set_option('display.max_columns', None)


    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    flag = True
    while flag:

        # data
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True

        # keep track of all choices made by the user
        all_choices = []
        while additional_category_flag:
            
            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            for i, option in enumerate(access_categories, start=1):
                print(f"{i}. {option}")
            print()
            print()

            # user input for main category #
            category = choose_from_list(access_categories,
                "Enter name or number of MRI finding: ",
                "Invalid input. Choose a category number or exact name from the list above.")

            if category is None:
                return "Forced exit."
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    for i, option in enumerate(access_subcategories, start=1):
                        print(f"{i}. {option}")
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = choose_from_list(access_subcategories,
                    "Choose a subcategory by name or number: ",
                    "Invalid input. Choose a subcategory number or exact name from the list above.")
                
                    if subcategory is None:
                        return "Forced exit."

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    for i, option in enumerate(access_sub_subcategories, start=1):
                        print(f"{i}. {option}")
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = choose_from_list(
                    access_sub_subcategories,"Choose a sub-subcategory by name or number: ",
                    "Invalid input. Choose a sub-subcategory number or exact name from the list above.")
                
                    if sub_subcategory is None:
                        return "Forced exit."
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    for i, option in enumerate(access_sub_subcategories, start=1):
                        print(f"{i}. {option}")
                    print()
                    print()

                    # user input for subcategory #
                    sub_subcategory = choose_from_list(access_sub_subcategories,
                    "Choose a sub-subcategory by name or number: ",
                    "Invalid input. Choose a sub-subcategory number or exact name from the list above.")
                
                    if sub_subcategory is None:
                        return "Forced exit."

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()

            # verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())
                        
            # display list of values of interest
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = choose_from_list(access_values_of_interest,
            "Choose a value of interest by name or number: ",
            "Invalid input. Choose a value number or exact name from the list above.")
            
            if value_of_interest is None:
                return "Forced exit."

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df[findings_df[col_title].str.contains(pat=r'\b{}\b'.format(value_of_interest), regex=True)]

            # update list of user choices #
            all_choices.append(str(col_title+"%"+value_of_interest))
            
            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # highlight relevant rows
        mask = df.index.isin(findings_df.index.tolist())
        result_rows = findings_df.index.tolist()
        def highlight_rows(row):
            if row.name in result_rows:
                return ['background-color: steelblue'] * len(row)
            else:
                return [''] * len(row)

        # display main results
        result_to_display = pd.DataFrame(df.loc[result_rows]).drop(["Clinical Synopsis"], axis=1)
        results = result_to_display.style.apply(highlight_rows, axis=1)
        display(results)
        print()

        # display rest of df #
        rest_of_data = df[~mask]
        rest_of_data["Rank"] = 0
        for i in range(len(all_choices)):
            col = all_choices[i].split('%')[0]
            value = all_choices[i].split('%')[1]
            rest_of_data.loc[rest_of_data[col]==value, 'Rank'] += 1
        rest_of_data = pd.DataFrame(rest_of_data.sort_values(by='Rank', ascending=False))
        display(pd.DataFrame(rest_of_data))

        # prompt user to start new search #
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [7]:
main()

List of categories:
*******************

1. Diffuse or Multifocal
2. Symmetric
3. Frontal vs Posterior Predominance
4. Telltale Grey Matter Involvement
5. Cortical Involvement/Subcortical Lesions
6. U-Fiber/Juxtacortical Involvement
7. Ventriculomegaly vs Hydrocephalus
8. Spinal Cord Involvement
9. Periventricular Involvement
10. Subcortical Structures




Enter name or number of MRI finding:  1



- You chose: [ Diffuse or Multifocal ].




List of Values of Interest:
***************************

D

M



Choose a value of interest by name or number:  1



- You chose: [ D ].


...

,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes
1,SPG2,"D, M",asymmetric,posterior,,,,ear of lynx,present,present,present,,,,,,"middle cerebellar peduncles, infratentorial lesions",pons,present,,,PLP1,"X:103,776,506-103,792,619",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chrX:103776506-103792619&dgv=pack&knownGene=pack&omimGene=pack,Complicated,XLR,C,"<1/1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/",https://doi.org/10.1016/j.jns.2017.01.069,# 312920 https://www.omim.org/entry/312920?search=spg2&highlight=spg2,"https://www.orpha.net/consor/cgi-bin/OC_Exp.php?lng=EN&Expert=99015#:~:text=Pure%20SPG2%20manifests%20as%20early,gait%20due%20to%20spastic%20paraparesis.",,Multifocal - MS like lesions
15,SPG47,D,"symmetric, asymmetric",posterior,,"cortical malformation, BPP",,"ear of lynx, ears of grizzly",ears of grizzly,heterotopia,,,present,severe atrophy,severe atrophy,"short CC, hemigenesis",mild atrophy of vermis,,,,ventriculomegaly,AP4B1,"1:113,894,194-113,905,028",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr1:113894194-113905028&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , https://doi.org/10.1002/ajmg.a.38561 , 10.1212/WNL.0000000000012836 , https://doi.org/10.1016/j.jns.2011.03.011 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514",# 614066 https://www.omim.org/entry/614066?search=spg47&highlight=spg47,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg47&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellectual%20disability%20and%20progressive%20spastic%20paraplegia&search=Disease_Search_Simple,,
18,SPG50,D,asymmetric,posterior,,"global white matter atrophy, vascular tortuosity",,present,present,present,,,,present,present,short CC,present,,,,asymmetric ventriculomegaly and CSF issue,AP4M1,"7:100,100,794-100,109,039",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&position=chr7:100100794-100109039&dgv=pack&knownGene=pack&omimGene=pack,Complicated,AR,"I, NN","<1 / 1,000,000","10.3389/fneur.2018.01117 , 10.1002/ajmg.a.36514 , https://www.ncbi.nlm.nih.gov/books/NBK1509/","10.1093/brain/awz307 , 10.1212/WNL.0000000000012836 , https://doi-org.proxy3.library.mcgill.ca/10.1002/ajmg.a.36514 , 10.1016/j.ajhg.2009.06.004 , https://doi.org/10.1212%2FNXG.0000000000000217",# 612936 https://www.omim.org/entry/612936?search=spg50&highlight=spg50,https://www.orpha.net/consor/cgi-bin/Disease_Search.php?lng=EN&data_id=20498&Disease_Disease_Search_diseaseGroup=spg50&Disease_Disease_Search_diseaseType=Pat&Disease(s)/group%20of%20diseases=Severe-intellectual-disability-and-progressive-spastic-paraplegia&title=Severe%20intellec

C:\Users\aley2\AppData\Local\Temp\ipykernel_16000\927888150.py:235: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rest_of_data["Rank"] = 0


,Gene,Diffuse or Multifocal,Symmetric,Frontal vs Posterior Predominance,Telltale Grey Matter Involvement,Cortical Involvement/Subcortical Lesions,U-Fiber/Juxtacortical Involvement,Periventricular Involvement:Forceps Major,Periventricular Involvement:Forceps Minor,Periventricular Involvement:Other,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Rostrum,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Genu,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Body,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Isthmus,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Splenium,Subcortical Structures:Corpus Callosum Involvement (Atrophy):Other,Subcortical Structures:Posterior Fossa Involvement:Cerebellar,Subcortical Structures:Posterior Fossa Involvement:Brainstem,Spinal Cord Involvement:Spinal Cord Atrophy,Spinal Cord Involvement:Abnormal Signal/White Matter Tract,Ventriculomegaly vs Hydrocephalus,Gene/Locus,Position,Genome Browser,Pure vs Complicated,MOI,Onset,Prevalence,Clinical Synopsis,Articles,Articles with MR Images,OMIM,Orpha link,Other,Bracket notes,Rank
0,SPG1,D,,,,,,,,,agenesis,agenesis,,,,,,,,,hydrocephalus,L1CAM,"X:153,861,516-153,886,173",https://www.ncbi.nlm.nih.gov/gene/3897,Complicated,XLR,NN,"1/300,000 Male",GROWTH\nHeight\n- Short stature (<5-15th perce...,"https://pubmed.ncbi.nlm.nih.gov/8305964/ , 10....",https://doi.org/10.1016/S0387-7604(97)00079-X,# 303350 https://www.omim.org/entry/303350,https://www.orpha.net/consor/cgi-bin/OC_Exp.ph...,,,1
9,SPG15,D,symmetric,frontal,,,,ear of lynx,present,,,present,present,present,,,mild atrophy of vermis,,rare,,,ZFYVE26,"14:67,728,892-67,816,590",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing deficit\nEyes\n- ...,"https://doi.org/10.1016/j.ajhg.2008.03.004 , h...","10.3389/fneur.2018.01117 , https://doi.org/10....",# 270700 https://www.omim.org/entry/270700?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,,1
17,SPG49,D,symmetric,,globus pallida calcifications,,,ear of lynx,present,,,,present,,,,,,,,,TECPR2,,,Complicated,AR,I,"<1 / 1,000,000",,"10.3389/fneur.2018.01117 , https://www.ncbi.nl...",https://doi.org/10.1016/j.ajhg.2012.11.001,,https://www.orpha.net/consor/cgi-bin/Disease_S...,Now called HSAN9,,1
14,SPG46,D,,posterior,,,,,,present,,,present,present,,,atrophy of vermis,,present,,ventriculomegaly,GBA2,"9:35,736,866-35,749,228",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"I, C","<1 / 1,000,000",HEAD & NECK\nEars\n- Hearing loss (in some pat...,"10.3389/fneur.2018.01117 , 10.1016/j.ajhg.2012...","https://doi.org/10.1007/s10048-010-0249-2 , 10...",# 614409 https://www.omim.org/entry/614409?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,,1
13,SPG35,D,symmetric,posterior,T2 hypointense pallidal nuclei,,,,ears of grizzly,present,,present,present,present,,,"severe atrophy of cerebellum, atrophy of vermis",,,,ventriculomegaly,FA2H,"16:74,712,969-74,774,820",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD","<1 / 1,000,000",HEAD & NECK\nEyes\n- External ophthalmoplegia ...,https://www.ncbi.nlm.nih.gov/books/NBK1509/,"10.3389/fneur.2018.01117 , https://doi.org/10....",# 612319 https://www.omim.org/entry/612319?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,,,1
12,SPG21,D,symmetric,posterior,,cortical atrophy,,,present,present,,present,present,present,,,mild atrophy of vermis,,,,,ACP33,"15:64,963,022-64,989,914",https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg...,Complicated,AR,"C, AD, A","<1 / 1,000,000",HEAD & NECK\nMouth\n- Brisk jaw jerk\nABDOMEN\...,"10.3389/fneur.2018.01117 , 10.1055/s-2005-8728...",10.1111/cns.13723,# 248900 https://www.omim.org/entry/248900?sea...,https://www.orpha.net/consor/cgi-bin/Disease_S...,Also called Mast Syndrome,,1
11,SPG20,D,symmetric,posterior,,corona radiata,,,mild,present,,,,,,,atrophy of superior vermis,,,,ventriculomegaly,SPART,"13:

Would you like to search again?  no


Goodbye!
